IMPORTAÇÃO DAS BIBLIOTECAS

* **Pandas:** trabalha com dados estruturados, principalmente tabelas.
* **Numpy:** trabalha com números, arrays e cálculos matemáticos de forma eficiente.

In [10]:
import pandas as pd
import numpy as np

CONFIGURAÇÕES

* Usaremos o módulo `np.random` para gerar dados aleatórios que comporão a base, respeitando a quantidade de colaboradores definida.
> A função `seed (x)` define a semente para a geração dos números pseudoaleatórios. O x pode ser qualquer número. Seeds diferentes geram sequências diferentes, permitindo criar bases aleatórias distintas e reproduzir cada uma posteriormente.

In [11]:
np.random.seed(40)

QUANTIDADE_COLABORADORES = 500

BASE DE DADOS

Para a criação da base de dados temos como itens obrigatórios:
- ID_Colaborador
- Data_Admissao
- Data_Desligamento
- Motivo_Desligamento

Lógica usada para criação dos dados:
- 500 colaboradores criados, para manter a mesma numeração de colaboradores da Base de Colaboradores.
- Uma data de admissão para cada colaborador, representando a data de início do vínculo.
- np.random.randint define aleatoriamente uma quantidade entre 140 e 160 colaboradores para representar registros de desligamento.
- np.random.choice seleciona os colaboradores de forma aleatória.
- O parâmetro replace=False impede que o mesmo colaborador seja selecionado mais de uma vez.

In [12]:
df = pd.DataFrame()

# ID
df["ID_Colaborador"] = [
    f"COL{i:05d}"
    for i in range(1, QUANTIDADE_COLABORADORES + 1)
]

# Data Admissão
df["Data_Admissao"] = pd.to_datetime(
    np.random.choice(
        pd.date_range(
            start="2021-01-01",
            end="2026-01-01"
        ),
        QUANTIDADE_COLABORADORES
    )
)

# Quantidade de desligamentos
quantidade_desligados = np.random.randint(
    140,
    161
)

# Seleção dos colaboradores desligados
desligados = np.random.choice(
    df["ID_Colaborador"],
    quantidade_desligados,
    replace=False
)

DESLIGAMENTOS

Lógica usada para criação dos desligamentos:
- `pd.NaT` representa a ausência de uma informação de data. Utilizamos esse valor inicialmente para que os registros sem data de desligamento permaneçam vazios.
- `np.random.choice` define aleatoriamente qual situação será aplicada à data de desligamento de cada colaborador selecionado.
- O `parâmetro p` define a probabilidade de ocorrência de cada situação dentro das opções disponíveis.
>Para a lógica das datas de desligamento, utilizamos as probabilidades para manter a maior parte dos registros em um cenário esperado, mas também gerar situações de ausência ou inconsistência que poderão ser identificadas posteriormente na análise da qualidade dos dados.

- As probabilidades devem somar 1 (100%) - Por isso: 0.85 + 0.05 + 0.05 + 0.05 = 1.00

In [13]:
df["Data_Desligamento"] = pd.NaT
df["Motivo_Desligamento"] = pd.NA

for idx in df.index[df["ID_Colaborador"].isin(desligados)]:

    tipo_data = np.random.choice(
        [
            "normal",
            "sem_data",
            "mesma_admissao",
            "antes_admissao"
        ],
        p=[0.85, 0.05, 0.05, 0.05]
    )

    if tipo_data == "normal":
        df.loc[idx, "Data_Desligamento"] = (
            df.loc[idx, "Data_Admissao"]
            + pd.Timedelta(
                days=np.random.randint(90, 1500)
            )
        )

    elif tipo_data == "mesma_admissao":
        df.loc[idx, "Data_Desligamento"] = (
            df.loc[idx, "Data_Admissao"]
        )

    elif tipo_data == "antes_admissao":
        df.loc[idx, "Data_Desligamento"] = (
            df.loc[idx, "Data_Admissao"]
            - pd.Timedelta(
                days=np.random.randint(1, 365)
            )
        )

MOTIVOS DOS DESLIGAMENTOS
- A lista motivos contém as opções de motivos de desligamento que poderão ser utilizadas na base.
- `None`, `"N/A"` e `""` são adicionados às opções para simular diferentes formas de ausência ou preenchimento inadequado da informação.
- `np.random.choice` seleciona aleatoriamente os motivos que serão atribuídos aos colaboradores desligados.
- O parâmetro `p` define a probabilidade de ocorrência de cada opção.
>Para a distribuição dos motivos, utilizamos probabilidades diferentes para tornar a base mais próxima de um cenário organizacional, mantendo os motivos mais comuns com maior probabilidade de ocorrência e os demais com menor frequência.

- As probabilidades devem somar 1 (100%) - Por isso: 0.70 + 0.07 + 0.06 + 0.05 + 0.04 + 0.03 + 0.02 + 0.01 + 0.01 + 0.01 = 1.00.



In [14]:
motivos = [
    "Pedido de demissão",
    "Desempenho",
    "Reestruturação",
    "Fim de contrato",
    "Mudança de carreira",
    "Salário",
    "Outro"
]

opcoes_motivos = motivos + [
    None,
    "N/A",
    ""
]

probabilidades = [
    0.70,
    0.07,
    0.06,
    0.05,
    0.04,
    0.03,
    0.02,
    0.01,
    0.01,
    0.01
]

df.loc[
    df["ID_Colaborador"].isin(desligados),
    "Motivo_Desligamento"
] = np.random.choice(
    opcoes_motivos,
    quantidade_desligados,
    p=probabilidades
)

CENÁRIO DE TESTE PARA SIMULAR ERRO DE PREENCHIMENTO
- `astype("string")` define o tipo da coluna Motivo_Desligamento como texto.
- `df["ID_Colaborador"].isin(desligados)` identifica os colaboradores que não foram selecionados como desligados.
- `np.random.random` gera valores aleatórios para definir quais registros receberão um preenchimento indevido.
- O valor `0.02` representa uma probabilidade de 2% de ocorrência desse preenchimento.
- O operador `&` combina as duas condições: o colaborador não foi selecionado como desligado e foi sorteado para receber um preenchimento.
- `np.random.choice` seleciona aleatoriamente um dos motivos disponíveis para preencher esses registros.
>O objetivo desse cenário é simular um erro de preenchimento em que um colaborador sem registro de desligamento possui um motivo de desligamento informado, criando uma inconsistência que poderá ser identificada e tratada posteriormente.

In [15]:
df["Motivo_Desligamento"] = df["Motivo_Desligamento"].astype("string")

nao_desligados = ~df["ID_Colaborador"].isin(desligados)

preenchimento_aleatorio = (
    np.random.random(len(df)) < 0.02
) & nao_desligados

df.loc[
    preenchimento_aleatorio,
    "Motivo_Desligamento"
] = np.random.choice(
    motivos,
    preenchimento_aleatorio.sum()
)

SCHEMA DO DATAFRAME
- ID_Colaborador → string
- Data_Admissao → datetime
- Data_Desligamento → datetime
- Motivo_Desligamento → string
>O schema define os tipos de dados esperados para cada coluna, garantindo uma estrutura consistente para utilização posterior nas análises e na exportação da base.

In [ ]:
df["ID_Colaborador"] = df["ID_Colaborador"].astype("string")

df["Data_Admissao"] = pd.to_datetime(
    df["Data_Admissao"]
)

df["Data_Desligamento"] = pd.to_datetime(
    df["Data_Desligamento"]
)

df["Motivo_Desligamento"] = df[
    "Motivo_Desligamento"
].astype("string")

EXPORTANDO PARA EXCEL

A exportação do documento final será feita de forma simplificada, sendo gerada e armazenada dentro das pastas do próprio notebook. Posteriormente, o arquivo será enviado manualmente para o repositório do projeto.

In [16]:
nome_arquivo = "base2_desligamentos.xlsx"

df.to_excel(
    nome_arquivo,
    index=False,
    sheet_name="Desligamentos"
)

print(
    f"Base criada com sucesso: {nome_arquivo}"
)

Base criada com sucesso: base2_desligamentos.xlsx


CONSIDERAÇÕES FINAIS

Nesta segunda versão, a base foi estruturada para representar informações de admissões e desligamentos, mantendo os mesmos colaboradores da Base 1 e incorporando diferentes cenários para avaliação da qualidade dos dados.
Entre os principais aprimoramentos implementados nesta versão:
- **Estruturação do schema do DataFrame**, definindo os tipos de dados esperados para cada campo.
- **Inclusão de cenários de dados ausentes**, utilizando NULL, N/A, valores vazios e NaT.
- **Simulação de inconsistências nas datas**, incluindo desligamentos na mesma data ou anteriores à admissão.
- **Simulação de erros de preenchimento**, incluindo motivos de desligamento registrados para colaboradores sem registro de desligamento.
- **Distribuição probabilística dos cenários**, mantendo predominância de registros esperados e introduzindo situações de ausência e inconsistência para posterior análise.
>Esta versão amplia a estrutura de dados do projeto e introduz diferentes cenários de qualidade de dados, preparando a base para a etapa de análise e tratamento das inconsistências.